# ***Importando bibliotecas***

In [189]:
# Importação de bibliotecas

import csv
import json
from datetime import datetime



# ***Criação de uma constante para limite suspeito***

In [190]:
#Verificar movimentações acima do limite definido
LIMITE_SUSPEITO = 10000.00

# ***Função para ler o arquivo sem validar os dados***

In [191]:
def ler_transacoes():

  try:

    with open("transacoes.csv", mode="r", encoding="utf-8") as arquivo:
      leitor = csv.DictReader(arquivo)
      transacoes = list(leitor)
      return transacoes

  except FileNotFoundError:

    print("Arquivo transacoes.csv não encontrado.")

    return[]

# **Função para verificar se os dados estão corretos.**

In [192]:
def validar_transacao(transacao):

  # Validar ID
  if not transacao["id"].isdigit():
    return None

  # Validar cliente
  if not transacao["cliente_id"]:
    return None

  # Validar tipo
  if transacao["tipo"] not in ["credito", "debito"]:
    return None

  # Validar valor
  try:
    valor = float(transacao["valor"])

    if valor <=0:
      return None

  except ValueError:
        return None

  # Validar data
  try:
    data = datetime.strptime(transacao["data"], "%Y-%m-%d")

  except ValueError:
    return None


  # Salvar valores convertidos
  transacao["valor"]  = valor
  transacao["data"]   = data

  return transacao


# ***Processar todas as transações***

In [193]:
def processar_transacoes():

  dados = ler_transacoes()

  transacoes_validas = []
  transacoes_invalidas = []

  for transacao in dados:
    resultado = validar_transacao(transacao)

    if resultado:
      transacoes_validas.append(resultado)

    else:
      transacoes_invalidas.append(transacao)


  print("===== RESUMO DA LIMPEZA =====")
  print()
  print(f"Total de linhas lidas: {len(dados)}")
  print(f"Linhas válidas: {len(transacoes_validas)}")
  print(f"Linhas inválidas: {len(transacoes_invalidas)}")


  return transacoes_validas, transacoes_invalidas





# ***Executar a função processar_transacões***

In [194]:
#Execução para teste
validas, invalidas = processar_transacoes()



===== RESUMO DA LIMPEZA =====

Total de linhas lidas: 24
Linhas válidas: 18
Linhas inválidas: 6


# ***Função para analisar periodo***

In [195]:
def analisar_periodo(transacoes):

  if not transacoes:
    print("Nenhuma transação válida encontrada.")
    return

  datas = []

  for transacao in transacoes:
    datas.append(transacao["data"])

  data_mais_antiga = min(datas)
  data_mais_recente = max(datas)

  dias = (data_mais_recente - data_mais_antiga).days

  print("\n\n===== PERÍODO ANALISADO ===== ")
  print(f"\nData mais antiga: {data_mais_antiga.strftime('%d/%m/%y')} ")
  print(f"Data mais recente: {data_mais_recente.strftime('%d/%m/%y')} ")
  print(f"Periodo analisado: {dias} dias ")


# ***Testanto resultado da análise***

In [196]:
#Execução para teste
validas, invalidas = processar_transacoes()

analisar_periodo(validas)

===== RESUMO DA LIMPEZA =====

Total de linhas lidas: 24
Linhas válidas: 18
Linhas inválidas: 6


===== PERÍODO ANALISADO ===== 

Data mais antiga: 05/01/26 
Data mais recente: 18/04/26 
Periodo analisado: 103 dias 


# ***Gerar relatório por mês***

In [197]:
def gerar_relatorio(transacoes):

  # Dicionário onde será armazenado o relatório por mês
  resumo = {}

  # Percorre todas as transações válidas
  for transacao in transacoes:

    # Extrai o mês da data no formato AAAA-MM
    mes = transacao["data"].strftime("%Y-%m")

    # Se o mês ainda não existir no resumo, cria sua estrutura
    if mes not in resumo:

      resumo[mes] = {

          "quantidade": 0,
          "total_credito": 0,
          "total_debito": 0,
          "saldo": 0,
          "soma": 0,
          "maior_valor": transacao["valor"],
          "menor_valor": transacao["valor"]
      }

    # Atualiza a quantidade de transações do mês
    resumo[mes]["quantidade"] +=1

    # Soma todos os valores do mês (será usada para calcular a média)
    resumo[mes]["soma"] += transacao["valor"]

    # Verifica se é crédito ou débito
    if transacao["tipo"] == "credito":
        resumo[mes]["total_credito"] += transacao["valor"]

    else:
        resumo[mes]["total_debito"] += transacao["valor"]

    # Atualiza o maior valor encontrado no mês
    if transacao["valor"] > resumo[mes]["maior_valor"]:
        resumo[mes]["maior_valor"] = transacao["valor"]

    # Atualiza o menor valor encontrado no mês
    if transacao["valor"] < resumo[mes]["menor_valor"]:
        resumo[mes]["menor_valor"] = transacao["valor"]

# Percorremos os meses para calcular saldo e média
  for mes in resumo:

    # Saldo = créditos - débitos
    resumo[mes]["saldo"] = (
        resumo[mes]["total_credito"] -
        resumo[mes]["total_debito"]
    )

    # Média = soma dos valores / quantidade de transações
    resumo[mes]["media"] = (
        resumo[mes]["soma"] /
        resumo[mes]["quantidade"]

    )

    # Remove o campo "soma", utilizado como um valor auxiliar
    del resumo[mes]["soma"]

  # Retorna o relatório final
  return resumo


# ***Identificar transações suspeitas acima de 10000***

In [198]:
def transacoes_suspeitas(transacoes):

  suspeitas=[]

  for transacao in transacoes:
    if transacao["valor"] > LIMITE_SUSPEITO:
      suspeitas.append(transacao)

  print("\n===== TRANSAÇÕES SUSPEITAS =====")
  print()
  print(f"Quantidade encontrada: {len(suspeitas)}")

  if len(suspeitas) == 0:
    print("Nenhuma transação suspeita encontrada")

  else:
   for transacao in suspeitas:

     print(
        f"ID: {transacao["id"]} | "
        f"Cliente: {transacao["cliente_id"]} | "
        f"Data: {transacao["data"].strftime('%d/%m/%Y')} | "
        f"Valor: R${transacao["valor"]:.2f}"
    )

  return suspeitas


## ***Testanto resultado da análise***

In [199]:
#Execução para teste
validas, invalidas = processar_transacoes()
suspeitas = transacoes_suspeitas(validas)

===== RESUMO DA LIMPEZA =====

Total de linhas lidas: 24
Linhas válidas: 18
Linhas inválidas: 6

===== TRANSAÇÕES SUSPEITAS =====

Quantidade encontrada: 2
ID: 7 | Cliente: CLI003 | Data: 14/02/2026 | Valor: R$15000.00
ID: 13 | Cliente: CLI002 | Data: 15/03/2026 | Valor: R$12000.00


# ***Criação de relatorios json***

In [200]:
def exportar_json(relatorio, suspeitas):

   resultado = {
      "relatorio_mensal": relatorio,
      "transacoes_suspeitas": suspeitas
   }

   with open("relatorio.json", "w", encoding="utf-8") as arquivo:

      json.dump(
        resultado,
        arquivo,
        ensure_ascii=False,
        indent=4,
        default=str
      )

   print("\nArquivo relatorio.json criado com sucesso!")

In [201]:
def imprimir_relatorio(relatorio):

    print("\n===== RELATÓRIO MENSAL =====")

    for mes, dados in relatorio.items():

        print(f"\nMês: {mes}")
        print(f"Quantidade........: {dados['quantidade']}")
        print(f"Créditos..........: R$ {dados['total_credito']:.2f}")
        print(f"Débitos...........: R$ {dados['total_debito']:.2f}")
        print(f"Saldo.............: R$ {dados['saldo']:.2f}")
        print(f"Média.............: R$ {dados['media']:.2f}")
        print(f"Maior Valor.......: R$ {dados['maior_valor']:.2f}")
        print(f"Menor Valor.......: R$ {dados['menor_valor']:.2f}")

# **Função principal para execução do código completo**

In [202]:
def main():

    validas, invalidas = processar_transacoes()

    analisar_periodo(validas)

    relatorio = gerar_relatorio(validas)

    imprimir_relatorio(relatorio)

    suspeitas = transacoes_suspeitas(validas)

    exportar_json(relatorio, suspeitas)





# **Ativação da função principal**

In [203]:
#Chamada de função para execução do código por completo
main()

===== RESUMO DA LIMPEZA =====

Total de linhas lidas: 24
Linhas válidas: 18
Linhas inválidas: 6


===== PERÍODO ANALISADO ===== 

Data mais antiga: 05/01/26 
Data mais recente: 18/04/26 
Periodo analisado: 103 dias 

===== RELATÓRIO MENSAL =====

Mês: 2026-01
Quantidade........: 5
Créditos..........: R$ 6300.00
Débitos...........: R$ 545.50
Saldo.............: R$ 5754.50
Média.............: R$ 1369.10
Maior Valor.......: R$ 3500.00
Menor Valor.......: R$ 45.00

Mês: 2026-02
Quantidade........: 5
Créditos..........: R$ 18500.00
Débitos...........: R$ 619.90
Saldo.............: R$ 17880.10
Média.............: R$ 3823.98
Maior Valor.......: R$ 15000.00
Menor Valor.......: R$ 99.90

Mês: 2026-03
Quantidade........: 5
Créditos..........: R$ 15500.00
Débitos...........: R$ 750.65
Saldo.............: R$ 14749.35
Média.............: R$ 3250.13
Maior Valor.......: R$ 12000.00
Menor Valor.......: R$ 99.90

Mês: 2026-04
Quantidade........: 3
Créditos..........: R$ 3200.00
Débitos...........: R$ 2